# ShotGuide Video Inference Pipeline

This notebook runs the current CLIP embedding model on a real mp4.

Pipeline:

1. Detect cuts with the existing adaptive frame-difference approach
2. Save scene metadata with start/end frame and timestamps
3. Extract multiple frames per scene
4. Encode frames with frozen CLIP ViT-B/32
5. Mean-pool frame embeddings into one scene embedding
6. Predict `shot_type` and `has_text`
7. Save `scene_predictions.csv`
8. Render a simple overlay video

In [1]:
from pathlib import Path
import json
import math

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image, ImageDraw, ImageFont
from tqdm.auto import tqdm
import open_clip

ROOT = Path.cwd()
VIDEO_PATH = ROOT / 'videos' / '0001_DXWQRkxS2SK.mp4'
CHECKPOINT_PATH = ROOT / 'outputs_clip_embeddings' / 'clip_vit_b32_multitask_head.pt'
OUTPUT_ROOT = ROOT / 'outputs_video_inference'

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert VIDEO_PATH.exists(), VIDEO_PATH
assert CHECKPOINT_PATH.exists(), CHECKPOINT_PATH
DEVICE

C:\Users\user\anaconda3\envs\shotguide-baseline\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device(type='cpu')

## 1. Cut Detection with Scene Metadata

In [2]:
def get_video_info(video_path: Path):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise ValueError(f'Cannot open video: {video_path}')
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()
    return {'fps': fps, 'frame_count': frame_count, 'width': width, 'height': height, 'duration_sec': frame_count / fps}

def calculate_frame_diffs(video_path: Path, resize_size=(320, 180), sample_interval=1):
    cap = cv2.VideoCapture(str(video_path))
    prev_gray = None
    diffs = []
    frame_idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if frame_idx % sample_interval != 0:
            frame_idx += 1
            continue
        resized = cv2.resize(frame, resize_size)
        gray = cv2.cvtColor(resized, cv2.COLOR_BGR2GRAY)
        if prev_gray is not None:
            diffs.append(float(np.mean(cv2.absdiff(prev_gray, gray))))
        prev_gray = gray
        frame_idx += 1
    cap.release()
    return np.array(diffs, dtype=np.float32)

def get_adaptive_threshold(video_path: Path, percentile=98, fallback_threshold=50, min_threshold=15, max_threshold=120):
    diffs = calculate_frame_diffs(video_path)
    if len(diffs) == 0:
        return float(fallback_threshold)
    threshold = np.percentile(diffs, percentile)
    return float(np.clip(threshold, min_threshold, max_threshold))

def detect_cut_frames(video_path: Path, percentile=98, min_scene_sec=0.2, resize_size=(320, 180)):
    info = get_video_info(video_path)
    fps = info['fps']
    min_scene_len = max(1, int(fps * min_scene_sec))
    threshold = get_adaptive_threshold(video_path, percentile=percentile)

    cap = cv2.VideoCapture(str(video_path))
    ret, prev_frame = cap.read()
    if not ret:
        cap.release()
        raise ValueError(f'Cannot read first frame: {video_path}')

    prev_gray = cv2.cvtColor(cv2.resize(prev_frame, resize_size), cv2.COLOR_BGR2GRAY)
    frame_idx = 0
    last_cut_frame = 0
    cut_frames = [0]

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame_idx += 1
        gray = cv2.cvtColor(cv2.resize(frame, resize_size), cv2.COLOR_BGR2GRAY)
        diff_score = float(np.mean(cv2.absdiff(prev_gray, gray)))
        enough_gap = (frame_idx - last_cut_frame) >= min_scene_len
        if diff_score > threshold and enough_gap:
            cut_frames.append(frame_idx)
            last_cut_frame = frame_idx
        prev_gray = gray

    cap.release()
    return cut_frames, threshold, info

def build_scene_table(video_path: Path, cut_frames, threshold, info):
    fps = info['fps']
    frame_count = info['frame_count']
    rows = []
    for i, start_frame in enumerate(cut_frames):
        next_start = cut_frames[i + 1] if i + 1 < len(cut_frames) else frame_count
        end_frame = max(start_frame, next_start - 1)
        rows.append({
            'video_path': str(video_path.resolve()),
            'video_name': video_path.name,
            'scene_index': i + 1,
            'start_frame': int(start_frame),
            'end_frame': int(end_frame),
            'start_time': start_frame / fps,
            'end_time': end_frame / fps,
            'duration_sec': (end_frame - start_frame + 1) / fps,
            'threshold': threshold,
            'fps': fps,
        })
    return pd.DataFrame(rows)

cut_frames, threshold, video_info = detect_cut_frames(VIDEO_PATH, percentile=98, min_scene_sec=0.2)
scene_df = build_scene_table(VIDEO_PATH, cut_frames, threshold, video_info)

print('video:', VIDEO_PATH)
print('info:', video_info)
print('threshold:', threshold)
print('scenes:', len(scene_df))
display(scene_df.head())

video: C:\Temp\deep\videos\0001_DXWQRkxS2SK.mp4
info: {'fps': 23.976, 'frame_count': 255, 'width': 720, 'height': 1280, 'duration_sec': 10.635635635635635}
threshold: 15.0
scenes: 4


,video_path,video_name,scene_index,start_frame,end_frame,start_time,end_time,duration_sec,threshold,fps
0,C:\Temp\deep\videos\0001_DXWQRkxS2SK.mp4,0001_DXWQRkxS2SK.mp4,1,0,35,0.000000,1.459793,1.501502,15.0,23.976
1,C:\Temp\deep\videos\0001_DXWQRkxS2SK.mp4,0001_DXWQRkxS2SK.mp4,2,36,94,1.501502,3.920587,2.460794,15.0,23.976
2,C:\Temp\deep\videos\0001_DXWQRkxS2SK.mp4,0001_DXWQRkxS2SK.mp4,3,95,141,3.962296,5.880881,1.960294,15.0,23.976
3,C:\Temp\deep\videos\0001_DXWQRkxS2SK.mp4,0001_DXWQRkxS2SK.mp4,4,142,254,5.922589,10.593927,4.713046,15.0,23.976


## 2. Extract Multiple Frames per Scene

In [3]:
def sample_frame_indices(start_frame, end_frame, num_samples=3):
    if end_frame <= start_frame:
        return [int(start_frame)]
    positions = np.linspace(0.20, 0.80, num_samples)
    indices = [int(round(start_frame + (end_frame - start_frame) * p)) for p in positions]
    return sorted(set(max(start_frame, min(end_frame, idx)) for idx in indices))

def read_frame_at(cap, frame_idx):
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_idx))
    ret, frame = cap.read()
    if not ret:
        return None
    return frame

def extract_scene_frames(video_path: Path, scene_df: pd.DataFrame, output_dir: Path, num_samples=3):
    output_dir.mkdir(parents=True, exist_ok=True)
    cap = cv2.VideoCapture(str(video_path))
    rows = []
    for _, scene in tqdm(scene_df.iterrows(), total=len(scene_df)):
        frame_indices = sample_frame_indices(int(scene.start_frame), int(scene.end_frame), num_samples=num_samples)
        frame_paths = []
        for j, frame_idx in enumerate(frame_indices, start=1):
            frame = read_frame_at(cap, frame_idx)
            if frame is None:
                continue
            frame_path = output_dir / f"scene_{int(scene.scene_index):03d}_frame_{j:02d}_{frame_idx:06d}.jpg"
            cv2.imwrite(str(frame_path), frame)
            frame_paths.append(str(frame_path.resolve()))
        row = scene.to_dict()
        row['sampled_frame_indices'] = json.dumps(frame_indices)
        row['sampled_frame_paths'] = json.dumps(frame_paths, ensure_ascii=False)
        rows.append(row)
    cap.release()
    return pd.DataFrame(rows)

video_output_dir = OUTPUT_ROOT / VIDEO_PATH.stem
frames_dir = video_output_dir / 'scene_frames'
scene_df = extract_scene_frames(VIDEO_PATH, scene_df, frames_dir, num_samples=3)
video_output_dir.mkdir(parents=True, exist_ok=True)
scene_df.to_csv(video_output_dir / 'scene_metadata.csv', index=False, encoding='utf-8-sig')
display(scene_df[['scene_index', 'start_time', 'end_time', 'sampled_frame_indices']].head())

  0%|          | 0/4 [00:00<?, ?it/s]

 50%|█████     | 2/4 [00:00<00:00, 11.16it/s]

100%|██████████| 4/4 [00:00<00:00,  9.87it/s]

100%|██████████| 4/4 [00:00<00:00, 10.02it/s]

,scene_index,start_time,end_time,sampled_frame_indices
0,1,0.000000,1.459793,"[7, 18, 28]"
1,2,1.501502,3.920587,"[48, 65, 82]"
2,3,3.962296,5.880881,"[104, 118, 132]"
3,4,5.922589,10.593927,"[164, 198, 232]"


## 3. Load CLIP and the Multi-task Head

In [4]:
class ClipEmbeddingMultiTaskHead(nn.Module):
    def __init__(self, embedding_dim=512, hidden_dim=256, num_shot_classes=5, dropout=0.20):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(embedding_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.shot_head = nn.Linear(hidden_dim, num_shot_classes)
        self.text_head = nn.Linear(hidden_dim, 2)

    def forward(self, x):
        z = self.shared(x)
        return self.shot_head(z), self.text_head(z)

checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
shot_to_idx = checkpoint['shot_to_idx']
idx_to_shot = {int(k): v for k, v in checkpoint['idx_to_shot'].items()}

clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    checkpoint.get('clip_model', 'ViT-B-32'),
    pretrained=checkpoint.get('clip_pretrained', 'openai'),
    device=DEVICE,
)
clip_model.eval()
for p in clip_model.parameters():
    p.requires_grad = False

head = ClipEmbeddingMultiTaskHead(embedding_dim=512, hidden_dim=256, num_shot_classes=len(shot_to_idx)).to(DEVICE)
head.load_state_dict(checkpoint['model_state_dict'])
head.eval()

print('loaded:', CHECKPOINT_PATH)
print('shot labels:', idx_to_shot)

C:\Users\user\anaconda3\envs\shotguide-baseline\lib\site-packages\open_clip\factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


loaded: C:\Temp\deep\outputs_clip_embeddings\clip_vit_b32_multitask_head.pt
shot labels: {0: 'close-up', 1: 'medium', 2: 'object', 3: 'space', 4: 'wide'}


## 4. Scene-level CLIP Mean Pooling and Prediction

In [5]:
def encode_image_paths(image_paths):
    tensors = []
    for path in image_paths:
        image = Image.open(path).convert('RGB')
        tensors.append(clip_preprocess(image))
    if not tensors:
        return None
    batch = torch.stack(tensors).to(DEVICE)
    with torch.no_grad():
        features = clip_model.encode_image(batch)
        features = features / features.norm(dim=-1, keepdim=True)
        scene_feature = features.mean(dim=0, keepdim=True)
        scene_feature = scene_feature / scene_feature.norm(dim=-1, keepdim=True)
    return scene_feature.float()

def guide_text(shot_type, has_text):
    base = {
        'close-up': '피사체의 표정이나 디테일을 강조한 클로즈업 장면입니다.',
        'medium': '피사체와 주변 맥락을 함께 보여주는 미디엄 샷입니다.',
        'wide': '넓은 공간과 피사체 배치를 보여주는 와이드 샷입니다.',
        'object': '특정 제품이나 오브젝트가 중심이 되는 장면입니다.',
        'space': '공간의 분위기와 환경 정보가 중심이 되는 장면입니다.',
    }.get(shot_type, '장면 구도를 확인해야 하는 컷입니다.')
    if has_text:
        return base + ' 화면 내 텍스트 정보도 함께 강조됩니다.'
    return base + ' 텍스트보다 시각적 구도가 중심입니다.'

prediction_rows = []
scene_embeddings = []

for _, scene in tqdm(scene_df.iterrows(), total=len(scene_df)):
    image_paths = json.loads(scene['sampled_frame_paths'])
    scene_feature = encode_image_paths(image_paths)
    if scene_feature is None:
        continue

    with torch.no_grad():
        shot_logits, text_logits = head(scene_feature)
        shot_prob = torch.softmax(shot_logits, dim=1).cpu().numpy()[0]
        text_prob = torch.softmax(text_logits, dim=1).cpu().numpy()[0]

    pred_shot_idx = int(np.argmax(shot_prob))
    pred_text_idx = int(np.argmax(text_prob))
    pred_shot = idx_to_shot[pred_shot_idx]
    has_text = bool(pred_text_idx == 1)

    row = scene.to_dict()
    row.update({
        'pred_shot_type': pred_shot,
        'pred_has_text': int(has_text),
        'shot_confidence': float(shot_prob[pred_shot_idx]),
        'text_probability': float(text_prob[1]),
        'guide_text': guide_text(pred_shot, has_text),
    })
    for i, label in idx_to_shot.items():
        row[f'prob_shot_{label}'] = float(shot_prob[i])
    prediction_rows.append(row)
    scene_embeddings.append(scene_feature.cpu().numpy()[0])

pred_df = pd.DataFrame(prediction_rows)
pred_csv = video_output_dir / 'scene_predictions.csv'
pred_df.to_csv(pred_csv, index=False, encoding='utf-8-sig')
np.savez_compressed(video_output_dir / 'scene_clip_embeddings.npz', embeddings=np.vstack(scene_embeddings).astype('float32'))

print('saved:', pred_csv)
display(pred_df[['scene_index', 'start_time', 'end_time', 'pred_shot_type', 'pred_has_text', 'shot_confidence', 'text_probability', 'guide_text']])

  0%|          | 0/4 [00:00<?, ?it/s]

 25%|██▌       | 1/4 [00:00<00:00,  6.21it/s]

 50%|█████     | 2/4 [00:00<00:00,  6.52it/s]

 75%|███████▌  | 3/4 [00:00<00:00,  6.55it/s]

100%|██████████| 4/4 [00:00<00:00,  6.58it/s]

100%|██████████| 4/4 [00:00<00:00,  6.53it/s]

saved: C:\Temp\deep\outputs_video_inference\0001_DXWQRkxS2SK\scene_predictions.csv


,scene_index,start_time,end_time,pred_shot_type,pred_has_text,shot_confidence,text_probability,guide_text
0,1,0.000000,1.459793,medium,1,0.968403,0.999193,피사체와 주변 맥락을 함께 보여주는 미디엄 샷입니다. 화면 내 텍스트 정보도 함께 ...
1,2,1.501502,3.920587,medium,1,0.884886,0.995872,피사체와 주변 맥락을 함께 보여주는 미디엄 샷입니다. 화면 내 텍스트 정보도 함께 ...
2,3,3.962296,5.880881,medium,1,0.901464,0.991759,피사체와 주변 맥락을 함께 보여주는 미디엄 샷입니다. 화면 내 텍스트 정보도 함께 ...
3,4,5.922589,10.593927,medium,1,0.966399,0.999241,피사체와 주변 맥락을 함께 보여주는 미디엄 샷입니다. 화면 내 텍스트 정보도 함께 ...


## 5. Render Overlay Video

In [6]:
def load_korean_font(size=28):
    candidates = [
        Path('C:/Windows/Fonts/malgun.ttf'),
        Path('C:/Windows/Fonts/malgunbd.ttf'),
        Path('C:/Windows/Fonts/arial.ttf'),
    ]
    for path in candidates:
        if path.exists():
            return ImageFont.truetype(str(path), size=size)
    return ImageFont.load_default()

def wrap_text(draw, text, font, max_width):
    words = text.split(' ')
    lines = []
    current = ''
    for word in words:
        candidate = word if not current else current + ' ' + word
        bbox = draw.textbbox((0, 0), candidate, font=font)
        if bbox[2] - bbox[0] <= max_width:
            current = candidate
        else:
            if current:
                lines.append(current)
            current = word
    if current:
        lines.append(current)
    return lines

def overlay_text_pil(frame_bgr, lines, font):
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    image = Image.fromarray(frame_rgb).convert('RGBA')
    overlay = Image.new('RGBA', image.size, (0, 0, 0, 0))
    draw = ImageDraw.Draw(overlay)

    margin = 24
    line_h = font.size + 8
    box_w = min(image.width - margin * 2, 760)
    box_h = margin + line_h * len(lines)
    draw.rounded_rectangle((margin, margin, margin + box_w, margin + box_h), radius=10, fill=(0, 0, 0, 170))

    y = margin + 12
    for line in lines:
        draw.text((margin + 16, y), line, font=font, fill=(255, 255, 255, 255))
        y += line_h

    out = Image.alpha_composite(image, overlay).convert('RGB')
    return cv2.cvtColor(np.array(out), cv2.COLOR_RGB2BGR)

def render_overlay_video(video_path: Path, pred_df: pd.DataFrame, output_path: Path):
    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    output_path.parent.mkdir(parents=True, exist_ok=True)
    writer = cv2.VideoWriter(str(output_path), cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))
    font = load_korean_font(size=max(22, int(height * 0.026)))

    scenes = pred_df.sort_values('start_frame').reset_index(drop=True)
    scene_idx = 0

    for frame_idx in tqdm(range(total_frames)):
        ret, frame = cap.read()
        if not ret:
            break

        while scene_idx + 1 < len(scenes) and frame_idx >= int(scenes.loc[scene_idx + 1, 'start_frame']):
            scene_idx += 1

        scene = scenes.loc[scene_idx]
        label = f"Scene {int(scene.scene_index):03d} | {scene.pred_shot_type} | text={int(scene.pred_has_text)} | shot {scene.shot_confidence:.2f} | text {scene.text_probability:.2f}"
        guide = str(scene.guide_text)

        temp_img = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        temp_draw = ImageDraw.Draw(temp_img)
        wrapped = [label] + wrap_text(temp_draw, guide, font, max_width=min(width - 80, 720))[:2]
        frame = overlay_text_pil(frame, wrapped, font)
        writer.write(frame)

    cap.release()
    writer.release()
    return output_path

overlay_path = video_output_dir / f'{VIDEO_PATH.stem}_overlay.mp4'
render_overlay_video(VIDEO_PATH, pred_df, overlay_path)
print('saved:', overlay_path)

  0%|          | 0/255 [00:00<?, ?it/s]

  2%|▏         | 4/255 [00:00<00:07, 33.90it/s]

  4%|▎         | 9/255 [00:00<00:06, 38.12it/s]

  5%|▌         | 13/255 [00:00<00:06, 37.79it/s]

  7%|▋         | 18/255 [00:00<00:06, 38.36it/s]

...

 96%|█████████▌| 245/255 [00:05<00:00, 48.12it/s]

 98%|█████████▊| 250/255 [00:05<00:00, 48.24it/s]

100%|██████████| 255/255 [00:05<00:00, 48.47it/s]

100%|██████████| 255/255 [00:05<00:00, 43.26it/s]

saved: C:\Temp\deep\outputs_video_inference\0001_DXWQRkxS2SK\0001_DXWQRkxS2SK_overlay.mp4
